# FATHOM - combine and scale

This notebook performs two steps:
1. Combine the three types of FATHOM flood (pluvial, fluvial, coastal) into a single layer of maximum flood depth.  
2. Scale the combined layer to match the 1km resolution GHS-POP layer, aggregating the data at the following classification  
  i. Proportion of 30m cells within each 1km grid cell where maximum flood depth exceeds 0.5m  
  ii. Proportion where maximum flood depth exceeds 0.15m  
  iii. Proportion where maximum flood depth exceeds 0m  

In [ ]:
import os, time, io, json, sys
import urllib3
import boto3
import rasterio

import geopandas as gpd
import pandas as pd
import numpy as np

from functools import reduce
from urllib3.exceptions import InsecureRequestWarning
from botocore import UNSIGNED
from botocore.config import Config
from tqdm.notebook import tqdm

urllib3.disable_warnings(InsecureRequestWarning)

def tPrint(s):
    """prints the time along with the message"""
    print("%s\t%s" % (time.strftime("%H:%M:%S"), s))

s3_client = boto3.client('s3', verify=False)

sys.path.insert(0, "C:/WBG/Work/Code/GOSTrocks/src")

import GOSTrocks.dataMisc as dataMisc
import GOSTrocks.rasterMisc as rMisc

%load_ext autoreload
%autoreload 2

In [ ]:
local_folder = "C:/WBG/Work/Projects/FATHOM_COLLAPSE"
out_folder = os.path.join(local_folder, "FATHOM_summaries")
map_folder = os.path.join(local_folder, "FATHOM_maps")
for tF in [out_folder, map_folder]:
    if not os.path.exists(tF):
        os.makedirs(tF)

ghs_pop_files = [
    "C:\\WBG\\Work\\data\\URBAN\\SMOD_POP\\GHS_POP_E2025_GLOBE_R2023A_54009_1000_V1_0.tif", 
    "C:\\WBG\\Work\\data\\URBAN\\SMOD_POP\\GHS_POP_E2030_GLOBE_R2023A_54009_100_V1_0.tif"
]

s3_bucket = "wbg-geography01"
s3_prefix = "FATHOM"
return_period = 100
flood_files = [
    ["FU", "FLOOD_MAP-1ARCSEC-NW_OFFSET-1in{rp}-FLUVIAL-UNDEFENDED-DEPTH-2020-PERCENTILE50-v3.1.vrt"],
    ["CU", "FLOOD_MAP-1ARCSEC-NW_OFFSET-1in{rp}-COASTAL-UNDEFENDED-DEPTH-2020-PERCENTILE50-v3.1.vrt"],
    ['PD', "FLOOD_MAP-1ARCSEC-NW_OFFSET-1in{rp}-PLUVIAL-DEFENDED-DEPTH-2020-PERCENTILE50-v3.1.vrt"]
]

In [ ]:
# Use the S3 client to get a list of VRT files in the specified bucket and prefix
response = s3_client.list_objects_v2(Bucket=s3_bucket, Prefix=s3_prefix)
vrt_files = [obj['Key'] for obj in response.get('Contents', []) if obj['Key'].endswith('.vrt')]

# Turn the list of vrt files into a dataframe
vrt_breakdown = [[x] + x.split('-') for x in vrt_files]
vrt_df = pd.DataFrame(vrt_breakdown, columns=["path", 'prefix', 'res', 'offset', 'return_period', 'hazard', 'defended', 'metric', 'year', 'scenario', 'version', "other"])
vrt_df = vrt_df.loc[:, ['return_period', 'hazard', 'defended', 'year', 'scenario', "path"]]

# Focus on just the 100-year return period for now
vrt_df = vrt_df[vrt_df['return_period'] == f"1in{return_period}"]
# Drop the undefended models
vrt_df = vrt_df[vrt_df['defended'] == "DEFENDED"]
vrt_df

In [ ]:
vrt_df.groupby(['year','scenario']).size()

In [ ]:
# Fetch the national boundaries from the official World Bank sources
iso3 = "GHA"
national_boundaries = dataMisc.download_wb_boundaries("ADM0", iso3)
national_boundaries.plot()

In [ ]:
# select the year and model to work with and extract the data from the vrt
year = "2020"
scenario = "PERCENTILE50"
depth_thresh = [0,15,50]
sel_models = vrt_df[(vrt_df['year'] == year) & (vrt_df['scenario'] == scenario)]

fluvial_path = "s3://{bucket}/{path}".format(bucket=s3_bucket, path=sel_models[sel_models['hazard'] == "FLUVIAL"]['path'].values[0])
coastal_path = "s3://{bucket}/{path}".format(bucket=s3_bucket, path=sel_models[sel_models['hazard'] == "COASTAL"]['path'].values[0])
pluvial_path = "s3://{bucket}/{path}".format(bucket=s3_bucket, path=sel_models[sel_models['hazard'] == "PLUVIAL"]['path'].values[0])


In [ ]:
tPrint("Starting")
for ghs_pop_file in ghs_pop_files:
    pop_file_name = ghs_pop_file.split("_")[-3]
    ghs_r = rasterio.open(ghs_pop_file)
    ghs_data, ghs_meta = rMisc.clipRaster(ghs_r, national_boundaries, None, True)
    tPrint(f"GHS population data clipped for {pop_file_name}m resolution")
    with rMisc.create_rasterio_inmemory(ghs_meta, ghs_data) as ghs_local:
        with rasterio.Env(GDAL_HTTP_UNSAFESSL='YES'):    
            fluvial_data, fluvial_meta = rMisc.clipRaster(fluvial_path, national_boundaries)
            coastal_data, coastal_meta = rMisc.clipRaster(coastal_path, national_boundaries)
            pluvial_data, pluvial_meta = rMisc.clipRaster(pluvial_path, national_boundaries)
            tPrint(f"FATHOM flood depth data clipped for fluvial, coastal, and pluvial hazards")
            # Stack the rasters together and take the max value across the stack to get the combined flood depth
            max_depth = np.maximum.reduce([fluvial_data, coastal_data, pluvial_data])
            for cDepth in depth_thresh:
                out_file = os.path.join(out_folder, f"{iso3}_FATHOM_combined_{year}_{scenario}_thresh{cDepth}_{pop_file_name}m_proportion.tif")
                if not os.path.exists(out_file):            
                    tPrint(f"Calculating combined flood depth for threshold {cDepth}m and population file {pop_file_name}...")
                    numerator = np.where(max_depth > cDepth, 1, 0)
                    denominator = np.where(max_depth > cDepth, 0, 1)
                    with rMisc.create_rasterio_inmemory(fluvial_meta, numerator[0,:,:]) as fathom_depth:
                        numerator_scaled, numerator_meta = rMisc.standardizeInputRasters(fathom_depth, ghs_local, resampling_type="sum")
                        tPrint(f"Numerator calculated and standardized: {numerator_scaled.dtype}")
                    with rMisc.create_rasterio_inmemory(fluvial_meta, denominator[0,:,:]) as fathom_depth:
                        denominator_scaled, denominator_meta = rMisc.standardizeInputRasters(fathom_depth, ghs_local, resampling_type="sum")
                        tPrint(f"Denominator calculated and standardized: {denominator_scaled.dtype}")
                    # write the denominator and numerator files as well for debugging purposes
                    with rasterio.open(out_file.replace("proportion", "numerator"), "w", **numerator_meta) as dest:
                        dest.write(numerator_scaled.astype(rasterio.float32))
                    with rasterio.open(out_file.replace("proportion", "denominator"), "w", **denominator_meta) as dest:
                        dest.write(denominator_scaled.astype(rasterio.float32))                                                           
                    results = numerator_scaled / (denominator_scaled + numerator_scaled)
                    numerator_meta.update({"dtype": rasterio.float32, "count": 1})                    
                    with rasterio.open(out_file, "w", **numerator_meta) as dest:
                        dest.write(results.astype(rasterio.float32))
                    tPrint(f"Output written to {out_file}")

            
                
            